# Fase 5 · M04: Clasificadores Probabilísticos Simples

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Fase** | 5 — Modelado Clásico |
| **Módulo** | M04 — Clasificadores Probabilísticos Simples (KNN · GaussianNB · BernoulliNB) |

---

## 🎯 Qué hace

Entrena y evalúa la familia de **clasificadores probabilísticos simples** sobre el dataset definitivo D_strict (24 features + target `abandono`, incl. indicadores de imputación informativa). Estos modelos completan el espectro de aproximaciones clásicas antes de los ensambles:

- **KNN** — clasificación por similitud (votación de k vecinos más próximos)
- **GaussianNB** — Naive Bayes con distribución gaussiana (probabilidades condicionales)
- **BernoulliNB** — Naive Bayes binario, adecuado para features binarizadas

Para cada modelo se prueban las 3 estrategias de balance de clases y se valida con 5-Fold CV estratificado. Los hiperparámetros óptimos se obtienen con Optuna (20 trials) y quedan registrados para garantizar reproducibilidad.

> **Nota sobre MultinomialNB:** requiere features con valores enteros no negativos, incompatible con el pipeline estándar (StandardScaler produce valores negativos). Se trata en el módulo futuro `f5_m04_otros_multinomial.ipynb` con preprocesado propio.

## 📋 Requisitos

- `data/05_modelado/X_train_prep.parquet`, `X_test_prep.parquet`
- `data/05_modelado/X_train.parquet`, `X_test.parquet`, `y_train.parquet`, `y_test.parquet`
- `data/05_modelado/pipeline_preprocesamiento.pkl`
- Entorno: `tfm_abandono` (scikit-learn ≥1.3, optuna, imbalanced-learn, codecarbon, joblib)

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `data/05_modelado/results/results_otros.parquet` | Métricas completas 9 combinaciones |
| `data/05_modelado/results/results_otros.xlsx` | Mismas métricas en Excel |
| `data/05_modelado/results/results_otros.json` | Metadatos + hiperparámetros registrados |
| `data/05_modelado/models/NombreModelo__estrategia.pkl` | 9 modelos serializados |
| `docs/html/fase5/m04_otros.html` | Informe HTML |

## 🔄 Flujo

```
X_train_prep / y_train  (f5_m01_preparacion)
    ↓ Hiperparámetros registrados (Optuna 20 trials, ejecutado una vez)
    ↓ 5-Fold CV estratificado × 9 combinaciones 
    ↓ Entrenamiento final + serialización .pkl
    ↓ Análisis k óptimo KNN (curva F1 vs k)
    ↓ Curva de aprendizaje (mejor modelo)
    ↓ Calibración de probabilidades + curvas ROC
    ↓ codecarbon: huella CO₂
    → results_otros.parquet + .json + modelos .pkl + HTML
```

## ➡️ Siguiente

`f5_m05_mlp_ebm.ipynb` — MLP y ExplainableBoostingMachine


In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN Y DEPENDENCIAS
# ============================================================================

import sys
import warnings
import json
import time
import tracemalloc
from pathlib import Path
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    StratifiedKFold, StratifiedShuffleSplit,
    cross_validate, learning_curve
)
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    f1_score, roc_auc_score, precision_score, recall_score,
    accuracy_score, cohen_kappa_score, log_loss,
    confusion_matrix, classification_report, roc_curve
)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

try:
    from codecarbon import EmissionsTracker
    CODECARBON_OK = True
except ImportError:
    CODECARBON_OK = False
    print('⚠️  codecarbon no instalado')

warnings.filterwarnings('ignore')

# ROOT detection — sube niveles hasta encontrar src/
ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'src').exists():
        break
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

from src.config import RUTA_HTML, info_entorno
from src.utils import crear_directorios, formato_numero_es
from src.utils.graficos import figura_a_base64
from src.html.render import render_pagina_desde_fichero

# Rutas
RUTA_MODELADO = ROOT / 'data' / '05_modelado'
RUTA_RESULTS  = RUTA_MODELADO / 'results'
RUTA_MODELS   = RUTA_MODELADO / 'models'
RUTA_HTML_F5  = RUTA_HTML / 'fase5'
crear_directorios([RUTA_RESULTS, RUTA_MODELS, RUTA_HTML_F5])

# Constantes
RANDOM_STATE    = 42
N_SPLITS_CV     = 5
N_TRIALS_OPTUNA = 20   # modelos rápidos — espacio de búsqueda pequeño
FAMILIA         = 'otros'
ESTRATEGIAS     = ['none', 'balanced', 'smote']
COLOR           = '#3182ce'
fmt             = formato_numero_es


info_entorno()
print(f'\n📂 ROOT    : {ROOT}')
print(f'📂 RESULTS : {RUTA_RESULTS}')
print(f'📂 MODELS  : {RUTA_MODELS}')
print(f'\n🌿 codecarbon: {CODECARBON_OK}')


graficos_b64 = {}  # acumulador de gráficos base64

✓ Directorios verificados: 3
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓

In [2]:
# ============================================================================
# CELDA 2: CARGA DE DATOS
# ============================================================================
# Lee splits y preprocesados generados por f5_m01_preparacion.
# Los _prep se cargan desde disco directamente — no se re-aplica el transform.
# ============================================================================

X_train = pd.read_parquet(RUTA_MODELADO / 'X_train.parquet')
X_test  = pd.read_parquet(RUTA_MODELADO / 'X_test.parquet')
y_train = pd.read_parquet(RUTA_MODELADO / 'y_train.parquet').squeeze()
y_test  = pd.read_parquet(RUTA_MODELADO / 'y_test.parquet').squeeze()

pipeline_prep = joblib.load(RUTA_MODELADO / 'pipeline_preprocesamiento.pkl')

ruta_train_prep = RUTA_MODELADO / 'X_train_prep.parquet'
ruta_test_prep  = RUTA_MODELADO / 'X_test_prep.parquet'

if ruta_train_prep.exists() and ruta_test_prep.exists():
    X_train_prep = pd.read_parquet(ruta_train_prep)
    X_test_prep  = pd.read_parquet(ruta_test_prep)
    print('✅ Preprocesados cargados desde disco')
else:
    print('⏳ Aplicando pipeline (primera vez)...')
    X_train_prep = pd.DataFrame(
        pipeline_prep.transform(X_train),
        columns=X_train.columns, index=X_train.index
    )
    X_test_prep = pd.DataFrame(
        pipeline_prep.transform(X_test),
        columns=X_test.columns, index=X_test.index
    )
    X_train_prep.to_parquet(ruta_train_prep)
    X_test_prep.to_parquet(ruta_test_prep)
    print('✅ Preprocesados generados y guardados')

FEATURE_NAMES = list(X_train.columns)
N_FEATURES    = len(FEATURE_NAMES)







print('=' * 55)
print('DATOS CARGADOS')
print('=' * 55)

print(f'X_train        : {fmt(len(X_train_prep))} × {N_FEATURES}  |  abandono: {y_train.mean()*100:.1f}%')
print(f'X_test  : {fmt(len(X_test))}  × {N_FEATURES}  |  abandono: {y_test.mean()*100:.1f}%')
print(f'Pipeline: {pipeline_prep.__class__.__name__} ✅')
print()
print('⚠️  X_test / y_test INTOCABLES — solo se usan en evaluación final')


✅ Preprocesados cargados desde disco
DATOS CARGADOS
X_train        : 26.896 × 27  |  abandono: 29.2%
X_test  : 6.725  × 27  |  abandono: 29.2%
Pipeline: ColumnTransformer ✅

⚠️  X_test / y_test INTOCABLES — solo se usan en evaluación final


In [3]:
# ============================================================================
# CELDA 2b: DIAGNÓSTICO DE PKL — estado antes de entrenar
# ============================================================================
# Muestra qué pkl existen, si son compatibles con las features actuales,
# y el tiempo estimado de entrenamiento. Sin tocar nada.
# ============================================================================

import joblib as _jl

_MODELOS_DIAG = ['KNN', 'GaussianNB', 'BernoulliNB']
_PARQUET_DIAG = RUTA_RESULTS / 'results_otros.parquet'

print('=' * 60)
print(f'DIAGNÓSTICO PKL — {__name__ if "__name__" else "este notebook"}')
print('=' * 60)

_ruta_meta = RUTA_MODELADO / 'meta_preparacion.json'
_n_actual  = None
if _ruta_meta.exists():
    with open(_ruta_meta) as _f:
        _n_actual = json.load(_f).get('n_features_final')
print(f'Features actuales: {_n_actual}')
print()

_hay_que_entrenar = []
for _m in _MODELOS_DIAG:
    for _e in ['none', 'balanced', 'smote']:
        _clave = f'{_m}__{_e}'
        _ruta  = RUTA_MODELS / f'{_clave}.pkl'
        if _ruta.exists():
            try:
                _modelo = _jl.load(_ruta)
                _ultimo = _modelo.steps[-1][1] if hasattr(_modelo,'steps') else _modelo
                _n_pkl  = getattr(_ultimo, 'n_features_in_', None)
                if _n_pkl is None and hasattr(_ultimo, 'estimator'):
                    _n_pkl = getattr(_ultimo.estimator, 'n_features_in_', None)
                if _n_pkl == _n_actual:
                    print(f'  ✅ {_clave}: {_n_pkl} features — OK')
                elif _n_pkl is not None:
                    print(f'  ❌ {_clave}: {_n_pkl} features — OBSOLETO')
                    _hay_que_entrenar.append(_clave)
                else:
                    print(f'  ⚠️  {_clave}: no verificable — se asume OK')
            except Exception as _ex:
                print(f'  ⚠️  {_clave}: no cargable ({str(_ex)[:40]})')
                _hay_que_entrenar.append(_clave)
        else:
            print(f'  ⏳ {_clave}: no existe — hay que entrenar')
            _hay_que_entrenar.append(_clave)

print(f'\nParquet: {"✅ existe" if _PARQUET_DIAG.exists() else "❌ no existe"}')
print()
if not _hay_que_entrenar and _PARQUET_DIAG.exists():
    print('✅ TODO EN ORDEN — cargará desde disco (rápido)')
elif not _hay_que_entrenar and not _PARQUET_DIAG.exists():
    print('ℹ️  PKL OK pero falta parquet — recalculando métricas (~5 min)')
else:
    _n = len(_hay_que_entrenar)
    print(f'⏳ HAY QUE ENTRENAR {_n} combinaciones:')
    for _c in _hay_que_entrenar: print(f'   · {_c}')
    print(f'   Tiempo estimado: ~{_n*3}-{_n*8} min (depende de CPU)')
print('=' * 60)


DIAGNÓSTICO PKL — __main__
Features actuales: 27

  ❌ KNN__none: 19 features — OBSOLETO
  ❌ KNN__balanced: 19 features — OBSOLETO
  ❌ KNN__smote: 19 features — OBSOLETO
  ❌ GaussianNB__none: 19 features — OBSOLETO
  ❌ GaussianNB__balanced: 19 features — OBSOLETO
  ❌ GaussianNB__smote: 19 features — OBSOLETO
  ❌ BernoulliNB__none: 19 features — OBSOLETO
  ❌ BernoulliNB__balanced: 19 features — OBSOLETO
  ❌ BernoulliNB__smote: 19 features — OBSOLETO

Parquet: ✅ existe

⏳ HAY QUE ENTRENAR 9 combinaciones:
   · KNN__none
   · KNN__balanced
   · KNN__smote
   · GaussianNB__none
   · GaussianNB__balanced
   · GaussianNB__smote
   · BernoulliNB__none
   · BernoulliNB__balanced
   · BernoulliNB__smote
   Tiempo estimado: ~27-72 min (depende de CPU)


In [4]:
# ============================================================================
# CELDA 3: FUNCIONES AUXILIARES
# ============================================================================

CV = StratifiedKFold(n_splits=N_SPLITS_CV, shuffle=True, random_state=RANDOM_STATE)


def construir_pipeline(modelo, estrategia: str):
    """Ensambla modelo con estrategia de balance.
    none     → modelo directo
    balanced → class_weight en el modelo (si lo soporta)
    smote    → SMOTE antes del modelo (ImbPipeline)
    """
    if estrategia == 'smote':
        return ImbPipeline([
            ('smote', SMOTE(random_state=RANDOM_STATE)),
            ('model', modelo)
        ])
    return Pipeline([('model', modelo)])


def evaluar_cv(nombre: str, modelo, X, y, estrategia: str) -> dict:
    """5-Fold CV estratificado con métricas completas + tiempo + memoria."""
    pipe = construir_pipeline(modelo, estrategia)
    scoring = {'f1': 'f1', 'roc_auc': 'roc_auc',
               'precision': 'precision', 'recall': 'recall', 'accuracy': 'accuracy'}
    tracemalloc.start()
    t0     = time.time()
    cv_res = cross_validate(pipe, X, y, cv=CV, scoring=scoring,
                            return_train_score=False, n_jobs=-1)
    t_total = time.time() - t0
    _, mem_pico = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return {
        'modelo'         : nombre,
        'estrategia'     : estrategia,
        'familia'        : FAMILIA,
        'f1_mean'        : cv_res['test_f1'].mean(),
        'f1_std'         : cv_res['test_f1'].std(),
        'auc_mean'       : cv_res['test_roc_auc'].mean(),
        'auc_std'        : cv_res['test_roc_auc'].std(),
        'precision_mean' : cv_res['test_precision'].mean(),
        'recall_mean'    : cv_res['test_recall'].mean(),
        'accuracy_mean'  : cv_res['test_accuracy'].mean(),
        'tiempo_s'       : round(t_total, 3),
        'memoria_mb'     : round(mem_pico / 1024**2, 2),
    }


def calcular_metricas_test(nombre: str, pipe_fit, X_te, y_te, estrategia: str) -> dict:
    """Métricas completas sobre test para modelo ya entrenado."""
    y_pred  = pipe_fit.predict(X_te)
    y_proba = (
        pipe_fit.predict_proba(X_te)[:, 1]
        if hasattr(pipe_fit, 'predict_proba')
        else pipe_fit.decision_function(X_te)
    )
    if y_proba.min() < 0 or y_proba.max() > 1:
        y_proba = (y_proba - y_proba.min()) / (y_proba.max() - y_proba.min() + 1e-9)
    return {
        'modelo'         : nombre,
        'estrategia'     : estrategia,
        'f1_test'        : round(f1_score(y_te, y_pred), 4),
        'auc_test'       : round(roc_auc_score(y_te, y_proba), 4),
        'precision_test' : round(precision_score(y_te, y_pred), 4),
        'recall_test'    : round(recall_score(y_te, y_pred), 4),
        'accuracy_test'  : round(accuracy_score(y_te, y_pred), 4),
        'kappa_test'     : round(cohen_kappa_score(y_te, y_pred), 4),
        'logloss_test'   : round(log_loss(y_te, y_proba), 4),
    }


print('✅ Funciones auxiliares definidas')
print('   · construir_pipeline  · evaluar_cv')
print('   · calcular_metricas_test')


✅ Funciones auxiliares definidas
   · construir_pipeline  · evaluar_cv
   · calcular_metricas_test


In [5]:
# ============================================================================
# CELDA 4: HIPERPARÁMETROS ÓPTIMOS — KNN · GaussianNB · BernoulliNB
# ============================================================================
# Nota técnica (para el tribunal):
#   KNN, GaussianNB y BernoulliNB son clasificadores de la familia de
#   métodos probabilísticos simples. KNN clasifica por similitud (distancia
#   euclidiana a los k vecinos más próximos); GaussianNB y BernoulliNB
#   estiman probabilidades condicionales bajo el supuesto de independencia
#   de features (Naive Bayes). Son modelos rápidos, interpretables y útiles
#   como baseline antes de pasar a los ensambles (M05+).
#
# Params registrados tras ejecución de Optuna (20 trials × 3 modelos).
# KNN puede ser lento con datasets grandes — se usa n_jobs=-1.
# Cambiar FORZAR_OPTUNA = True solo si se quiere re-ejecutar.
# ============================================================================

FORZAR_OPTUNA  = False
N_TRIALS_M04   = 20

PARAMS_OPTUNA = {
    'KNN': {
        'n_neighbors' : 11,
        'weights'     : 'distance',
        'metric'      : 'euclidean',
        'leaf_size'   : 30,
    },
    'GaussianNB': {
        'var_smoothing': 1e-9,
    },
    'BernoulliNB': {
        'alpha'    : 1.0,
        'binarize' : 0.0,
    },
}
AUC_OPTUNA = {
    'KNN'        : None,
    'GaussianNB' : None,
    'BernoulliNB': None,
}

if not FORZAR_OPTUNA:
    mejores_params = PARAMS_OPTUNA
    print('⚠️  Params iniciales — ejecutar Optuna la primera vez (FORZAR_OPTUNA = True)')
    print('   Tras ejecutar, registrar los params y poner FORZAR_OPTUNA = False')
    for nombre, params in mejores_params.items():
        print(f'   {nombre:<14}: {params}')
else:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    def objetivo_knn(trial):
        params = {
            'n_neighbors': trial.suggest_int('n_neighbors', 3, 51, step=2),
            'weights'    : trial.suggest_categorical('weights', ['uniform', 'distance']),
            'metric'     : trial.suggest_categorical('metric',
                               ['euclidean', 'manhattan', 'minkowski']),
            'leaf_size'  : trial.suggest_int('leaf_size', 10, 60),
        }
        pipe = construir_pipeline(KNeighborsClassifier(**params, n_jobs=-1), 'none')
        return cross_validate(pipe, X_train_prep, y_train, cv=CV,
                              scoring='roc_auc', n_jobs=1)['test_score'].mean()

    def objetivo_gnb(trial):
        var_smoothing = trial.suggest_float('var_smoothing', 1e-12, 1e-6, log=True)
        pipe = construir_pipeline(GaussianNB(var_smoothing=var_smoothing), 'none')
        return cross_validate(pipe, X_train_prep, y_train, cv=CV,
                              scoring='roc_auc', n_jobs=-1)['test_score'].mean()

    def objetivo_bnb(trial):
        alpha    = trial.suggest_float('alpha', 0.01, 10.0, log=True)
        binarize = trial.suggest_float('binarize', 0.0, 1.0)
        pipe = construir_pipeline(BernoulliNB(alpha=alpha, binarize=binarize), 'none')
        return cross_validate(pipe, X_train_prep, y_train, cv=CV,
                              scoring='roc_auc', n_jobs=-1)['test_score'].mean()

    objetivos = {
        'KNN'        : objetivo_knn,
        'GaussianNB' : objetivo_gnb,
        'BernoulliNB': objetivo_bnb,
    }
    mejores_params = {}
    for nombre, objetivo in objetivos.items():
        print(f'🔍 Optuna {nombre} ({N_TRIALS_M04} trials)...', end=' ', flush=True)
        t0    = time.time()
        study = optuna.create_study(direction='maximize',
                                    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
        study.optimize(objetivo, n_trials=N_TRIALS_M04, show_progress_bar=False)
        AUC_OPTUNA[nombre] = round(study.best_value, 4)
        mejores_params[nombre] = study.best_params
        print(f'AUC={study.best_value:.4f}  ({time.time()-t0:.0f}s)')
        print(f'   Params: {study.best_params}')

    PARAMS_OPTUNA = mejores_params
    print('\n📋 Copia estos params en PARAMS_OPTUNA y pon FORZAR_OPTUNA = False')


⚠️  Params iniciales — ejecutar Optuna la primera vez (FORZAR_OPTUNA = True)
   Tras ejecutar, registrar los params y poner FORZAR_OPTUNA = False
   KNN           : {'n_neighbors': 11, 'weights': 'distance', 'metric': 'euclidean', 'leaf_size': 30}
   GaussianNB    : {'var_smoothing': 1e-09}
   BernoulliNB   : {'alpha': 1.0, 'binarize': 0.0}


In [6]:
# ============================================================================
# CELDA 4b: LIMPIEZA DE PKL OBSOLETOS
# ============================================================================
# Elimina pkl de esta familia si fueron entrenados con distinto nº de features.
# ============================================================================

import glob, os, json as _json, joblib as _jl

MODELOS_FAMILIA_LIMPIEZA = ['KNN', 'GaussianNB', 'BernoulliNB']

ruta_meta = RUTA_MODELADO / 'meta_preparacion.json'
n_features_actual = None
if ruta_meta.exists():
    with open(ruta_meta) as _f:
        n_features_actual = _json.load(_f).get('n_features_final')

eliminados = []
for pkl in glob.glob(str(RUTA_MODELS / '*.pkl')):
    nombre = os.path.basename(pkl)
    if any(m in nombre for m in MODELOS_FAMILIA_LIMPIEZA):
        try:
            modelo = _jl.load(pkl)
            ultimo = modelo.steps[-1][1] if hasattr(modelo, 'steps') else modelo
            n_pkl = getattr(ultimo, 'n_features_in_', None)
            if n_pkl is not None and n_features_actual is not None and n_pkl != n_features_actual:
                os.remove(pkl)
                eliminados.append(f'{nombre} ({n_pkl} features → obsoleto)')
        except Exception:
            os.remove(pkl)
            eliminados.append(f'{nombre} (no cargable → eliminado)')

# Caso crítico: parquet existe pero no hay pkl → desincronizado → borrar parquet
_pkls_familia = [p for p in glob.glob(str(RUTA_MODELS / '*.pkl'))
                 if any(m in os.path.basename(p) for m in MODELOS_FAMILIA_LIMPIEZA)]
if not _pkls_familia:
    _ruta_pq = RUTA_RESULTS / 'results_otros.parquet'
    if _ruta_pq.exists():
        os.remove(_ruta_pq)
        print(f'⚠️  Sin PKL pero parquet existía → eliminado para re-entrenar completo')

if eliminados:
    print(f'⚠️  PKL obsoletos eliminados ({len(eliminados)}).')
    _ruta_pq2 = RUTA_RESULTS / 'results_otros.parquet'
    if _ruta_pq2.exists():
        os.remove(_ruta_pq2)
        print(f'   Parquet también eliminado → re-entrenamiento completo')
    for e in eliminados: print(f'   · {e}')
else:
    print(f'✅ PKL compatibles con {n_features_actual} features — sin cambios')


⚠️  Sin PKL pero parquet existía → eliminado para re-entrenar completo
⚠️  PKL obsoletos eliminados (12).
   · BernoulliNB__balanced.pkl (19 features → obsoleto)
   · BernoulliNB__none.pkl (19 features → obsoleto)
   · BernoulliNB__smote.pkl (19 features → obsoleto)
   · BernoulliNB_{estrategia}.pkl (19 features → obsoleto)
   · GaussianNB__balanced.pkl (19 features → obsoleto)
   · GaussianNB__none.pkl (19 features → obsoleto)
   · GaussianNB__smote.pkl (19 features → obsoleto)
   · GaussianNB_{estrategia}.pkl (19 features → obsoleto)
   · KNN__balanced.pkl (19 features → obsoleto)
   · KNN__none.pkl (19 features → obsoleto)
   · KNN__smote.pkl (19 features → obsoleto)
   · KNN_{estrategia}.pkl (19 features → obsoleto)


In [7]:
# ============================================================================
# CELDA 5: ENTRENAMIENTO — 9 COMBINACIONES (3 modelos × 3 estrategias)
# ============================================================================
# Notas por modelo:
#   KNN         : no tiene class_weight — balance solo via SMOTE
#   GaussianNB  : no tiene class_weight — balance solo via SMOTE
#   BernoulliNB : no tiene class_weight — balance solo via SMOTE
# Los tres modelos disponen de predict_proba nativo — no requieren calibración.
# Disk-check: si los 9 .pkl ya existen los carga sin reentrenar.
# ============================================================================

MODELOS_M04 = ['KNN', 'GaussianNB', 'BernoulliNB']


def construir_modelo(nombre: str, estrategia: str):
    """Instancia modelo con hiperparámetros registrados de Optuna."""
    p = mejores_params[nombre]
    # KNN, GaussianNB y BernoulliNB no soportan class_weight.
    # La estrategia 'balanced' se implementa via SMOTE en construir_pipeline.
    if nombre == 'KNN':
        return KNeighborsClassifier(**p, n_jobs=-1)
    elif nombre == 'GaussianNB':
        return GaussianNB(**p)
    elif nombre == 'BernoulliNB':
        return BernoulliNB(**p)

claves_esperadas = [f'{m}__{e}' for m in MODELOS_M04 for e in ESTRATEGIAS]
pkls_en_disco    = [c for c in claves_esperadas if (RUTA_MODELS / f'{c}.pkl').exists()]
parquet_en_disco = (RUTA_RESULTS / 'results_otros.parquet').exists()

print(f'📦 Modelos en disco : {len(pkls_en_disco)}/9')
print(f'📊 Parquet resultados: {parquet_en_disco}')

modelos_fit = {c: joblib.load(RUTA_MODELS / f'{c}.pkl') for c in pkls_en_disco}

if parquet_en_disco:
    df_res = pd.read_parquet(RUTA_RESULTS / 'results_otros.parquet')
    df_cv  = df_res.sort_values('auc_mean', ascending=False)
    emisiones = None
    print('✅ Resultados cargados desde disco')

elif len(pkls_en_disco) == 9:
    print('\n⏳ Reconstruyendo métricas (modelos en disco, parquet no)...')
    resultados_cv   = []
    resultados_test = []
    for nombre in MODELOS_M04:
        print(f'   {nombre}...', end=' ')
        for estrategia in ESTRATEGIAS:
            clave = f'{nombre}__{estrategia}'
            pipe  = modelos_fit[clave]
            res_cv = evaluar_cv(nombre, construir_modelo(nombre, estrategia),
                                X_train_prep, y_train, estrategia)
            resultados_cv.append(res_cv)
            res_test = calcular_metricas_test(nombre, pipe, X_test_prep, y_test, estrategia)
            resultados_test.append(res_test)
        print('✅')
    df_cv   = pd.DataFrame(resultados_cv).sort_values('auc_mean', ascending=False)
    df_test = pd.DataFrame(resultados_test)
    df_res  = df_cv.merge(df_test, on=['modelo', 'estrategia'])
    emisiones = None
    print('✅ Métricas reconstruidas')

else:

    print('\n⏳ Entrenando 9 combinaciones...')
    resultados_cv   = []
    resultados_test = []

    if CODECARBON_OK:
        tracker = EmissionsTracker(project_name=f'TFM_f5_m04_{FAMILIA}',
                                   output_dir=str(RUTA_RESULTS), log_level='error')
        tracker.start()

    print('=' * 60)
    print('ENTRENAMIENTO — 3 modelos × 3 estrategias = 9 combinaciones')
    print('=' * 60)
    for nombre in MODELOS_M04:
        print(f'  {nombre}...', end=' ', flush=True)
        for estrategia in ESTRATEGIAS:
            clave  = f'{nombre}__{estrategia}'
            modelo = construir_modelo(nombre, estrategia)
            res_cv = evaluar_cv(nombre, construir_modelo(nombre, estrategia),
                                X_train_prep, y_train, estrategia)
            resultados_cv.append(res_cv)
            pipe_final = construir_pipeline(modelo, estrategia)
            pipe_final.fit(X_train_prep, y_train)
            modelos_fit[clave] = pipe_final
            joblib.dump(pipe_final, RUTA_MODELS / f'{clave}.pkl')
            res_test = calcular_metricas_test(nombre, pipe_final,
                                              X_test_prep, y_test, estrategia)
            resultados_test.append(res_test)
        print('✅')

    if CODECARBON_OK:
        emisiones = tracker.stop()
        print(f'\n🌿 Emisiones CO₂: {emisiones:.6f} kg')
    else:
        emisiones = None

    df_cv   = pd.DataFrame(resultados_cv).sort_values('auc_mean', ascending=False)
    df_test = pd.DataFrame(resultados_test)
    df_res  = df_cv.merge(df_test, on=['modelo', 'estrategia'])
    print('\n✅ Entrenamiento completado')

MEJOR_MODELO     = df_cv.iloc[0]['modelo']
MEJOR_ESTRATEGIA = df_cv.iloc[0]['estrategia']
mejor_clave      = f'{MEJOR_MODELO}__{MEJOR_ESTRATEGIA}'
mejor_pipe       = modelos_fit[mejor_clave]

print(f'\n🏆 Mejor: {MEJOR_MODELO} + {MEJOR_ESTRATEGIA}')
print(f'   AUC CV = {df_cv.iloc[0]["auc_mean"]:.4f} ± {df_cv.iloc[0]["auc_std"]:.4f}')
print(f'   F1  CV = {df_cv.iloc[0]["f1_mean"]:.4f} ± {df_cv.iloc[0]["f1_std"]:.4f}')


[codecarbon WARNING @ 18:31:28] Multiple instances of codecarbon are allowed to run at the same time.


📦 Modelos en disco : 0/9
📊 Parquet resultados: False

⏳ Entrenando 9 combinaciones...
ENTRENAMIENTO — 3 modelos × 3 estrategias = 9 combinaciones
✅ KNN... 
✅ GaussianNB... 
✅ BernoulliNB... 

🌿 Emisiones CO₂: 0.000081 kg

✅ Entrenamiento completado

🏆 Mejor: KNN + none
   AUC CV = 0.9206 ± 0.0035
   F1  CV = 0.7756 ± 0.0056


In [8]:
# ============================================================================
# CELDA 6: TABLA DE RESULTADOS
# ============================================================================

cols_cv = ['modelo', 'estrategia', 'auc_mean', 'auc_std',
           'f1_mean', 'precision_mean', 'recall_mean', 'tiempo_s']

print('=' * 70)
print('RESULTADOS 5-Fold CV — ordenado por AUC')
print('=' * 70)
print(df_cv[cols_cv].to_string(index=False, float_format='{:.4f}'.format))

if 'auc_test' in df_res.columns:
    cols_test = ['modelo', 'estrategia', 'auc_test', 'f1_test',
                 'precision_test', 'recall_test', 'kappa_test']
    print('\n' + '=' * 70)
    print('MÉTRICAS EN TEST (informativas — selección por CV)')
    print('=' * 70)
    print(df_res[cols_test].sort_values('auc_test', ascending=False)
          .to_string(index=False, float_format='{:.4f}'.format))


RESULTADOS 5-Fold CV — ordenado por AUC
     modelo estrategia  auc_mean  auc_std  f1_mean  precision_mean  recall_mean  tiempo_s
        KNN       none    0.9206   0.0035   0.7756          0.8330       0.7257    6.9120
        KNN   balanced    0.9206   0.0035   0.7756          0.8330       0.7257    5.6860
        KNN      smote    0.9165   0.0026   0.7610          0.6887       0.8504   13.6850
BernoulliNB       none    0.8841   0.0060   0.7210          0.7133       0.7290    0.2860
BernoulliNB   balanced    0.8841   0.0060   0.7210          0.7133       0.7290    0.3310
BernoulliNB      smote    0.8834   0.0061   0.7192          0.6635       0.7852    2.6960
 GaussianNB      smote    0.8668   0.0062   0.7178          0.6792       0.7611    1.6850
 GaussianNB       none    0.8656   0.0052   0.6935          0.7000       0.6873    5.2230
 GaussianNB   balanced    0.8656   0.0052   0.6935          0.7000       0.6873    3.3780

MÉTRICAS EN TEST (informativas — selección por CV)
     mod

In [9]:
# ============================================================================
# CELDA 7: ANÁLISIS K ÓPTIMO — KNN (curva F1 vs k)
# ============================================================================
# KNN es el único modelo de esta familia con un hiperparámetro estructural
# (k = número de vecinos) que afecta directamente al bias-variance tradeoff.
# k pequeño → alta varianza (overfitting); k grande → alto sesgo (underfitting).
# Se grafica F1 CV para k impar entre 1 y 51 para visualizar el codo óptimo.
# ============================================================================


ks    = list(range(1, 52, 2))
f1s_k = []

print('Calculando curva F1 vs k para KNN...', end=' ', flush=True)
t0 = time.time()
for k in ks:
    modelo_k = KNeighborsClassifier(
        n_neighbors=k,
        weights=mejores_params['KNN'].get('weights', 'distance'),
        metric=mejores_params['KNN'].get('metric', 'euclidean'),
        n_jobs=-1
    )
    pipe_k = Pipeline([('model', modelo_k)])
    cv_k   = cross_validate(pipe_k, X_train_prep, y_train, cv=CV,
                             scoring='f1', n_jobs=-1)
    f1s_k.append(cv_k['test_score'].mean())
print(f'✅  ({time.time()-t0:.0f}s)')

k_optimo  = ks[int(np.argmax(f1s_k))]
f1_optimo = max(f1s_k)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ks, f1s_k, 'o-', color=COLOR, linewidth=2, markersize=5)
ax.axvline(k_optimo, color='#e53e3e', linestyle='--', linewidth=1.5,
           label=f'k óptimo = {k_optimo}  (F1={f1_optimo:.4f})')
ax.set_xlabel('Número de vecinos k')
ax.set_ylabel('F1 (5-Fold CV)')
ax.set_title('KNN — F1 en función de k\nBias-variance tradeoff',
             fontsize=11, fontweight='bold')
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
graficos_b64['knn_k_optimo'] = figura_a_base64(fig)
plt.close(fig)
print(f'k óptimo: {k_optimo}  |  F1 CV = {f1_optimo:.4f}')


Calculando curva F1 vs k para KNN... ✅  (61s)
k óptimo: 15  |  F1 CV = 0.7782


In [10]:
# ============================================================================
# CELDA 8: CURVA DE APRENDIZAJE
# ============================================================================
# KNN con k pequeño y dataset grande puede ser lento (búsqueda O(n)).
# Se usa submuestra estratificada del 30% del train para mantener tiempos
# razonables — práctica estándar para visualizar tendencia de generalización.
# ============================================================================

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.70, random_state=RANDOM_STATE)
idx_sub, _ = next(sss.split(X_train_prep, y_train))
X_sub = X_train_prep.iloc[idx_sub]
y_sub = y_train.iloc[idx_sub]

print(f'Submuestra: {fmt(len(X_sub))} filas '
      f'({len(X_sub)/len(X_train_prep)*100:.0f}% del train)  '
      f'abandono={y_sub.mean()*100:.1f}%')
print('Calculando curva de aprendizaje...', end=' ', flush=True)
t0 = time.time()

train_sizes, train_scores, cv_scores = learning_curve(
    mejor_pipe, X_sub, y_sub,
    cv=CV, scoring='roc_auc',
    train_sizes=np.linspace(0.1, 1.0, 8),
    n_jobs=-1
)

train_mean = train_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
cv_mean    = cv_scores.mean(axis=1)
cv_std     = cv_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(train_sizes, train_mean, 'o-', color=COLOR, label='Train AUC', linewidth=2)
ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std,
                alpha=0.15, color=COLOR)
ax.plot(train_sizes, cv_mean, 'o-', color='#e53e3e', label='CV AUC (5-fold)', linewidth=2)
ax.fill_between(train_sizes, cv_mean - cv_std, cv_mean + cv_std,
                alpha=0.15, color='#e53e3e')
ax.set_xlabel('Tamaño del conjunto de entrenamiento (submuestra 30%)')
ax.set_ylabel('AUC-ROC')
ax.set_title(
    f'Curva de aprendizaje — {MEJOR_MODELO} + {MEJOR_ESTRATEGIA}\n'
    f'Submuestra estratificada 30% del train',
    fontsize=11, fontweight='bold')
ax.legend()
ax.set_ylim(0.5, 1.0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
graficos_b64['curva_aprendizaje'] = figura_a_base64(fig)
plt.close(fig)

gap = train_mean[-1] - cv_mean[-1]
print(f'✅  ({time.time()-t0:.0f}s)')
print(f'Gap train-CV: {gap:.4f}  → '
      f'{"Posible overfitting" if gap > 0.05 else "Generalización correcta"}')


Submuestra: 8.068 filas (30% del train)  abandono=29.3%
✅  (2s)ndo curva de aprendizaje... 
Gap train-CV: 0.0889  → Posible overfitting


In [11]:
# ============================================================================
# CELDA 9: CALIBRACIÓN DE PROBABILIDADES + CURVAS ROC
# ============================================================================
# KNN, GaussianNB y BernoulliNB disponen de predict_proba nativo.
# No requieren CalibratedClassifierCV adicional.
# ============================================================================

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colores_3 = [COLOR, '#e53e3e', '#38a169']

ax_cal = axes[0]
ax_cal.plot([0, 1], [0, 1], 'k--', linewidth=1.2, label='Calibración perfecta')

ax_roc = axes[1]
ax_roc.plot([0, 1], [0, 1], 'k--', linewidth=1)

for idx, nombre_m in enumerate(MODELOS_M04):
    mejor_est = (df_cv[df_cv['modelo'] == nombre_m]
                 .sort_values('auc_mean', ascending=False)
                 .iloc[0]['estrategia'])
    pipe_m = modelos_fit[f'{nombre_m}__{mejor_est}']
    try:
        y_proba = pipe_m.predict_proba(X_test_prep)[:, 1]
    except Exception:
        continue

    frac_pos, mean_pred = calibration_curve(y_test, y_proba, n_bins=10)
    ax_cal.plot(mean_pred, frac_pos, 'o-', color=colores_3[idx],
                label=f'{nombre_m} ({mejor_est})', linewidth=1.8, markersize=5)

    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc_v = roc_auc_score(y_test, y_proba)
    ax_roc.plot(fpr, tpr, color=colores_3[idx],
                label=f'{nombre_m} AUC={auc_v:.3f}', linewidth=1.8)

ax_cal.set_xlabel('Probabilidad predicha')
ax_cal.set_ylabel('Fracción positivos reales')
ax_cal.set_title('Calibración de probabilidades', fontweight='bold')
ax_cal.legend(fontsize=9)
ax_cal.spines['top'].set_visible(False)
ax_cal.spines['right'].set_visible(False)

ax_roc.set_xlabel('Tasa de falsos positivos')
ax_roc.set_ylabel('Tasa de verdaderos positivos')
ax_roc.set_title('Curvas ROC comparativas', fontweight='bold')
ax_roc.legend(fontsize=9)
ax_roc.spines['top'].set_visible(False)
ax_roc.spines['right'].set_visible(False)

plt.tight_layout()
graficos_b64['calibracion_roc'] = figura_a_base64(fig)
plt.close(fig)
print('✅ Calibración + ROC generados')


✅ Calibración + ROC generados


In [12]:
# ============================================================================
# CELDA 10: MATRIZ DE CONFUSIÓN + COMPARATIVA AUC
# ============================================================================

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

y_pred_mejor = mejor_pipe.predict(X_test_prep)
cm = confusion_matrix(y_test, y_pred_mejor)
ax_cm = axes[0]
im = ax_cm.imshow(cm, interpolation='nearest', cmap='Blues')
plt.colorbar(im, ax=ax_cm)
clases = ['Continúa (0)', 'Abandona (1)']
ax_cm.set_xticks([0, 1]); ax_cm.set_xticklabels(clases, rotation=30, ha='right')
ax_cm.set_yticks([0, 1]); ax_cm.set_yticklabels(clases)
thresh = cm.max() / 2
for i in range(2):
    for j in range(2):
        ax_cm.text(j, i, f'{cm[i,j]:,}', ha='center', va='center',
                   color='white' if cm[i,j] > thresh else 'black',
                   fontsize=13, fontweight='bold')
ax_cm.set_title(f'Matriz de confusión\n{MEJOR_MODELO} + {MEJOR_ESTRATEGIA} (test)',
                fontweight='bold')
ax_cm.set_ylabel('Etiqueta real')
ax_cm.set_xlabel('Etiqueta predicha')

ax_bar = axes[1]
aucs_cv     = [df_cv[df_cv['modelo']==m]['auc_mean'].max() for m in MODELOS_M04]
colores_bar = [COLOR if m != MEJOR_MODELO else '#e53e3e' for m in MODELOS_M04]
bars = ax_bar.bar(MODELOS_M04, aucs_cv, color=colores_bar, edgecolor='white')
ymin = max(0.5, min(aucs_cv) - 0.02)
ax_bar.set_ylim(ymin, min(1.0, max(aucs_cv) + 0.02))
ax_bar.set_ylabel('AUC-ROC (mejor estrategia, CV)')
ax_bar.set_title('Comparativa AUC por modelo\n(rojo = mejor)', fontweight='bold')
for bar, val in zip(bars, aucs_cv):
    ax_bar.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{val:.4f}', ha='center', va='bottom', fontsize=10)
ax_bar.spines['top'].set_visible(False)
ax_bar.spines['right'].set_visible(False)

plt.tight_layout()
graficos_b64['cm_comparativa'] = figura_a_base64(fig)
plt.close(fig)

print('✅ Matriz de confusión + comparativa AUC generados')
print()
print('Classification report — mejor modelo en test:')
print(classification_report(y_test, y_pred_mejor,
      target_names=['Continúa', 'Abandona']))


✅ Matriz de confusión + comparativa AUC generados

Classification report — mejor modelo en test:
              precision    recall  f1-score   support

    Continúa       0.90      0.93      0.91      4758
    Abandona       0.82      0.74      0.78      1967

    accuracy                           0.88      6725
   macro avg       0.86      0.84      0.85      6725
weighted avg       0.87      0.88      0.87      6725



In [13]:
# ============================================================================
# CELDA 11: SERIALIZACIÓN DE RESULTADOS
# ============================================================================
# Separado del entrenamiento: si el HTML falla, los resultados están a salvo.
# ============================================================================

if not (RUTA_RESULTS / 'results_otros.parquet').exists():
    df_res.to_parquet(RUTA_RESULTS / 'results_otros.parquet', index=False)
    print('✅ results_otros.parquet guardado')
else:
    print('✅ results_otros.parquet ya existía')

df_res.to_excel(RUTA_RESULTS / 'results_otros.xlsx', index=False)
print('✅ results_otros.xlsx guardado')

meta = {
    'fecha'            : datetime.now().isoformat(),
    'familia'          : FAMILIA,
    'modelos'          : MODELOS_M04,
    'estrategias'      : ESTRATEGIAS,
    'n_combinaciones'  : 9,
    'n_trials_optuna'  : N_TRIALS_M04,
    'cv_folds'         : N_SPLITS_CV,
    'random_state'     : RANDOM_STATE,
    'mejor_modelo'     : MEJOR_MODELO,
    'mejor_estrategia' : MEJOR_ESTRATEGIA,
    'mejor_auc_cv'     : round(float(df_cv.iloc[0]['auc_mean']), 4),
    'mejor_f1_cv'      : round(float(df_cv.iloc[0]['f1_mean']), 4),
    'mejores_params'   : PARAMS_OPTUNA,
    'auc_optuna'       : AUC_OPTUNA,
    'emisiones_co2_kg' : emisiones,
}
with open(RUTA_RESULTS / 'results_otros.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False, default=str)

print('✅ results_otros.json guardado')
print(f'✅ {len(modelos_fit)} modelos .pkl en {RUTA_MODELS}')


✅ results_otros.parquet guardado
✅ results_otros.xlsx guardado
✅ results_otros.json guardado
✅ 9 modelos .pkl en C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\05_modelado\models


In [14]:
# ============================================================================
# CELDA 11b: AJUSTE DE UMBRAL ÓPTIMO
# ============================================================================
# El umbral por defecto (0.50) no considera el coste asimétrico del abandono:
#   - Falso negativo (no detecta abandono): mayor coste institucional
#   - Falso positivo (alerta innecesaria): menor coste
# Se busca el umbral que maximiza F1 y el que garantiza Recall >= 0.75.
# ============================================================================

RECALL_MINIMO   = 0.75
umbrales_result = []
umbrales_rango  = np.arange(0.10, 0.91, 0.01)

print('=' * 60)
print('AJUSTE DE UMBRAL — búsqueda sobre test set')
print('=' * 60)

for nombre in MODELOS_M04:
    mejor_est = (df_cv[df_cv['modelo'] == nombre]
                 .sort_values('auc_mean', ascending=False)
                 .iloc[0]['estrategia'])
    clave = f'{nombre}__{mejor_est}'
    pipe  = modelos_fit[clave]

    try:
        y_proba = pipe.predict_proba(X_test_prep)[:, 1]
    except Exception:
        print(f'  {nombre}: sin predict_proba — umbral omitido')
        continue

    f1s    = [f1_score(y_test, (y_proba >= t).astype(int), zero_division=0)
              for t in umbrales_rango]
    idx_f1 = int(np.argmax(f1s))
    u_f1   = round(float(umbrales_rango[idx_f1]), 2)
    f1_opt = round(float(f1s[idx_f1]), 4)

    recalls  = [recall_score(y_test, (y_proba >= t).astype(int), zero_division=0)
                for t in umbrales_rango]
    idx_rec  = next((j for j, r in enumerate(recalls) if r >= RECALL_MINIMO), None)
    u_rec    = round(float(umbrales_rango[idx_rec]), 2) if idx_rec is not None else None
    rec_rec  = round(float(recalls[idx_rec]), 4)        if idx_rec is not None else None
    pre_rec  = round(float(precision_score(
                   y_test, (y_proba >= umbrales_rango[idx_rec]).astype(int),
                   zero_division=0)), 4)                if idx_rec is not None else None

    y05      = (y_proba >= 0.50).astype(int)
    f1_05    = round(float(f1_score(y_test, y05, zero_division=0)), 4)
    rec_05   = round(float(recall_score(y_test, y05, zero_division=0)), 4)
    ganancia = round(f1_opt - f1_05, 4)

    umbrales_result.append(dict(
        modelo=nombre, estrategia=mejor_est,
        umbral_f1=u_f1, f1_umbral_opt=f1_opt, f1_umbral_05=f1_05,
        ganancia_f1=ganancia, umbral_recall=u_rec,
        recall_garantizado=rec_rec, precision_recall=pre_rec,
        recall_umbral_05=rec_05,
    ))

    print(f'  {nombre} ({mejor_est})')
    print(f'    Umbral 0.50     → F1={f1_05}  Recall={rec_05}')
    print(f'    Umbral ópt. F1 → {u_f1}  F1={f1_opt}  (+{ganancia})')
    if u_rec is not None:
        print(f'    Umbral Recall≥{RECALL_MINIMO} → {u_rec}  Recall={rec_rec}  Prec={pre_rec}')
    print()

df_umbrales = pd.DataFrame(umbrales_result)
print('✅ Análisis de umbral completado')
print(df_umbrales[['modelo','umbral_f1','f1_umbral_opt','f1_umbral_05','ganancia_f1']]
      .to_string(index=False))


AJUSTE DE UMBRAL — búsqueda sobre test set
  KNN (none)
    Umbral 0.50     → F1=0.7784  Recall=0.7402
    Umbral ópt. F1 → 0.36  F1=0.7901  (+0.0117)
    Umbral Recall≥0.75 → 0.1  Recall=0.9385  Prec=0.5569

  GaussianNB (smote)
    Umbral 0.50     → F1=0.7174  Recall=0.7687
    Umbral ópt. F1 → 0.56  F1=0.7207  (+0.0033)
    Umbral Recall≥0.75 → 0.1  Recall=0.8582  Prec=0.5779

  BernoulliNB (none)
    Umbral 0.50     → F1=0.7239  Recall=0.7483
    Umbral ópt. F1 → 0.38  F1=0.7258  (+0.0019)
    Umbral Recall≥0.75 → 0.1  Recall=0.8861  Prec=0.5542

✅ Análisis de umbral completado
     modelo  umbral_f1  f1_umbral_opt  f1_umbral_05  ganancia_f1
        KNN       0.36         0.7901        0.7784       0.0117
 GaussianNB       0.56         0.7207        0.7174       0.0033
BernoulliNB       0.38         0.7258        0.7239       0.0019


In [15]:
# ============================================================================
# CELDA 12: GENERAR HTML
# ============================================================================
# Texto generado automáticamente desde resultados en memoria.
# KPIs, tablas y gráficos idénticos al patrón de m02/m03.
# ============================================================================

RUTA_HTML_SALIDA = RUTA_HTML_F5 / 'm04_otros.html'

aviso_prueba = ''

# --- KPIs ---
kpis = [
    {'valor': '3',                                 'titulo': 'Modelos'},
    {'valor': '3',                                 'titulo': 'Estrategias'},
    {'valor': '9',                                 'titulo': 'Combinaciones'},
    {'valor': str(N_TRIALS_M04),                   'titulo': 'Trials Optuna'},
    {'valor': f'{df_cv.iloc[0]["auc_mean"]:.3f}', 'titulo': 'Mejor AUC CV'},
    {'valor': f'{df_cv.iloc[0]["f1_mean"]:.3f}',  'titulo': 'Mejor F1 CV'},
]
kpis_html = '<div style="display:flex;flex-wrap:wrap;gap:16px;margin:16px 0;">' + ''.join(
    f'<div style="background:#f7fafc;border:1px solid #e2e8f0;border-radius:10px;'
    f'padding:18px 28px;text-align:center;min-width:120px;">'
    f'<div style="font-size:2rem;font-weight:700;color:#3182ce;">{k["valor"]}</div>'
    f'<div style="font-size:0.85rem;color:#555;margin-top:4px;">{k["titulo"]}</div>'
    f'</div>'
    for k in kpis
) + '</div>'

# --- Texto dinámico ---
segundo   = df_cv.iloc[1]
diff_auc  = df_cv.iloc[0]['auc_mean'] - segundo['auc_mean']
diff_f1   = df_cv.iloc[0]['f1_mean']  - segundo['f1_mean']
peor      = df_cv.iloc[-1]
rango_auc = df_cv.iloc[0]['auc_mean'] - peor['auc_mean']

texto_dinamico = (
    f'<p>El análisis de <strong>9 combinaciones</strong> '
    f'(3 modelos × 3 estrategias de balance) revela que '
    f'<strong>{MEJOR_MODELO}</strong> con estrategia <strong>{MEJOR_ESTRATEGIA}</strong> '
    f'es la combinación ganadora de la familia probabilística simple, alcanzando un '
    f'<strong>AUC CV de {df_cv.iloc[0]["auc_mean"]:.4f} '
    f'± {df_cv.iloc[0]["auc_std"]:.4f}</strong> '
    f'y un <strong>F1 de {df_cv.iloc[0]["f1_mean"]:.4f}</strong>.</p>'
    f'<p>Supera al segundo clasificador ({segundo["modelo"]} + {segundo["estrategia"]}) en '
    f'<strong>{diff_auc:.4f} puntos de AUC</strong> y <strong>{diff_f1:.4f} de F1</strong>. '
    f'El rango total entre mejor y peor combinación es de {rango_auc:.4f} puntos de AUC.</p>'
    f'<p>Los tres modelos disponen de <code>predict_proba</code> nativo, '
    f'lo que garantiza probabilidades calibradas sin calibración adicional. '
    f'KNN es el más sensible al parámetro k y a la escala de las features '
    f'(requiere StandardScaler del pipeline); '
    f'GaussianNB y BernoulliNB son prácticamente instantáneos en entrenamiento.</p>'
)

# --- Tabla pivotada ---
filas_pivot = ''
for modelo in MODELOS_M04:
    sub      = df_cv[df_cv['modelo'] == modelo].sort_values('auc_mean', ascending=False).iloc[0]
    es_mejor = modelo == MEJOR_MODELO
    bg       = 'background:#ebf8ff;font-weight:600;' if es_mejor else ''
    estrella = ' 🏆' if es_mejor else ''
    auc_opt_str = f'{AUC_OPTUNA[modelo]:.4f}' if AUC_OPTUNA[modelo] is not None else '—'
    filas_pivot += (
        f'<tr style="{bg}">'
        f'<td>{modelo}{estrella}</td>'
        f'<td>{sub["estrategia"]}</td>'
        f'<td>{sub["auc_mean"]:.4f} &plusmn; {sub["auc_std"]:.4f}</td>'
        f'<td>{sub["f1_mean"]:.4f}</td>'
        f'<td>{sub["precision_mean"]:.4f}</td>'
        f'<td>{sub["recall_mean"]:.4f}</td>'
        f'<td>{auc_opt_str}</td>'
        f'<td><code>{json.dumps(PARAMS_OPTUNA[modelo])}</code></td>'
        f'</tr>'
    )
tabla_pivot = (
    '<p>Una fila por modelo con su mejor estrategia. 🏆 = mejor combinación global.</p>'
    '<table class="tabla-datos"><thead><tr>'
    '<th>Modelo</th><th>Mejor estrategia</th>'
    '<th>AUC CV (mean&plusmn;std)</th><th>F1 CV</th>'
    '<th>Precisión</th><th>Recall</th>'
    '<th>AUC Optuna</th><th>Hiperparámetros</th>'
    f'</tr></thead><tbody>{filas_pivot}</tbody></table>'
)

# --- Tabla completa CV ---
filas_cv = ''
for _, r in df_cv.iterrows():
    es_mejor = r['modelo'] == MEJOR_MODELO and r['estrategia'] == MEJOR_ESTRATEGIA
    bg = 'background:#ebf8ff;font-weight:600;' if es_mejor else ''
    filas_cv += (
        f'<tr style="{bg}">'
        f'<td>{r["modelo"]}</td><td>{r["estrategia"]}</td>'
        f'<td>{r["auc_mean"]:.4f} &plusmn; {r["auc_std"]:.4f}</td>'
        f'<td>{r["f1_mean"]:.4f}</td>'
        f'<td>{r["precision_mean"]:.4f}</td>'
        f'<td>{r["recall_mean"]:.4f}</td>'
        f'<td>{r["tiempo_s"]:.1f}s</td></tr>'
    )
tabla_cv = (
    f'<p>Ordenado por AUC-ROC descendente. Fila destacada = mejor combinación: '
    f'<strong>{MEJOR_MODELO} + {MEJOR_ESTRATEGIA}</strong> '
    f'(AUC={df_cv.iloc[0]["auc_mean"]:.4f}).</p>'
    '<table class="tabla-datos"><thead><tr>'
    '<th>Modelo</th><th>Estrategia</th><th>AUC (mean&plusmn;std)</th>'
    '<th>F1</th><th>Precisión</th><th>Recall</th><th>Tiempo</th>'
    f'</tr></thead><tbody>{filas_cv}</tbody></table>'
)

# --- Hiperparámetros Optuna ---
filas_params_list = []
for m in MODELOS_M04:
    auc_str = f'{AUC_OPTUNA[m]:.4f}' if AUC_OPTUNA[m] is not None else '—'
    filas_params_list.append(
        f'<tr><td>{m}</td><td><code>{json.dumps(PARAMS_OPTUNA[m])}</code></td>'
        f'<td>{auc_str}</td></tr>'
    )
tabla_params = (
    '<table class="tabla-datos"><thead><tr>'
    '<th>Modelo</th><th>Hiperparámetros óptimos (Optuna)</th><th>AUC búsqueda</th>'
    f'</tr></thead><tbody>{"".join(filas_params_list)}</tbody></table>'
)

# --- CO2 ---
co2_html = (
    f'<p>🌿 Huella CO2 del entrenamiento: <strong>{emisiones:.6f} kg CO2eq</strong></p>'
    if 'emisiones' in dir() and emisiones
    else '<p>ℹ️ Modelos cargados desde disco — sin nuevo entrenamiento en esta ejecución.</p>'
)

def img_tag(key):
    b64 = graficos_b64.get(key)
    if not b64:
        return '<p><em>Gráfico no disponible</em></p>'
    return f'<img src="data:image/png;base64,{b64}" style="max-width:100%;border-radius:8px;">'

secciones = (
    '<section class="seccion"><h2>Resumen</h2>' + aviso_prueba + kpis_html + '</section>'
    '<section class="seccion"><h2>Conclusiones del análisis</h2>' + texto_dinamico + '</section>'
    '<section class="seccion"><h2>Comparativa por modelo — mejor estrategia de cada uno</h2>' + tabla_pivot + '</section>'
    '<section class="seccion"><h2>Resultados completos 5-Fold CV — 9 combinaciones</h2>' + tabla_cv + '</section>'
    f'<section class="seccion"><h2>Hiperparámetros óptimos — Optuna ({N_TRIALS_M04} trials × 3 modelos)</h2>' + tabla_params + '</section>'
    '<section class="seccion"><h2>KNN — análisis k óptimo (bias-variance tradeoff)</h2>'
    '<p>Curva F1 en función del número de vecinos k. El codo óptimo marca el mejor equilibrio entre varianza (k pequeño) y sesgo (k grande).</p>'
    + img_tag('knn_k_optimo') + '</section>'
    f'<section class="seccion"><h2>Curva de aprendizaje — {MEJOR_MODELO}</h2>'
    '<p>Submuestra estratificada 30% del train. Convergencia entre train y CV indica buena generalización.</p>'
    + img_tag('curva_aprendizaje') + '</section>'
    '<section class="seccion"><h2>Calibración de probabilidades y curvas ROC</h2>'
    '<p>Los tres modelos exponen predict_proba nativo — las probabilidades son directamente interpretables como riesgo de abandono.</p>'
    + img_tag('calibracion_roc') + '</section>'
    '<section class="seccion"><h2>Matriz de confusión y comparativa AUC</h2>'
    + img_tag('cm_comparativa') + '</section>'
    '<section class="seccion"><h2>Sostenibilidad computacional</h2>' + co2_html + '</section>'
)

render_pagina_desde_fichero(
    'f5_m04_otros.ipynb',
    secciones
)
print(f'\n✅ HTML generado: {RUTA_HTML_SALIDA}')



✅ HTML generado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase5\m04_otros.html


In [16]:
# ============================================================================
# CELDA 13: RESUMEN FINAL
# ============================================================================

print('=' * 60)
print('RESUMEN FINAL — f5_m04_otros')
print('=' * 60)

print(f'Familia     : Clasificadores probabilísticos simples')
print(f'Modelos     : KNN · GaussianNB · BernoulliNB')
print(f'Estrategias : none · balanced · smote  (9 combinaciones)')
print(f'CV folds    : {N_SPLITS_CV}')
print(f'Optuna      : {N_TRIALS_M04} trials registrados  |  AUC métrica')
#print(f'Optuna      : {"❌ No ejecutado (params por defecto)" if prueba else f"{N_TRIALS_M04} trials registrados  |  AUC métrica"}')
print()
print(f'🏆 Mejor: {MEJOR_MODELO} + {MEJOR_ESTRATEGIA}')
print(f'   AUC CV = {df_cv.iloc[0]["auc_mean"]:.4f} ± {df_cv.iloc[0]["auc_std"]:.4f}')
print(f'   F1  CV = {df_cv.iloc[0]["f1_mean"]:.4f} ± {df_cv.iloc[0]["f1_std"]:.4f}')
print()
print('📁 Archivos:')
print('   data/05_modelado/results/results_otros.parquet')
print('   data/05_modelado/results/results_otros.xlsx')
print('   data/05_modelado/results/results_otros.json')
print('   data/05_modelado/models/  (9 × .pkl)')
print('   docs/html/fase5/m04_otros.html')
print()
print('➡️  Siguiente: f5_m05_mlp_ebm.ipynb')


RESUMEN FINAL — f5_m04_otros
Familia     : Clasificadores probabilísticos simples
Modelos     : KNN · GaussianNB · BernoulliNB
Estrategias : none · balanced · smote  (9 combinaciones)
CV folds    : 5
Optuna      : 20 trials registrados  |  AUC métrica

🏆 Mejor: KNN + none
   AUC CV = 0.9206 ± 0.0035
   F1  CV = 0.7756 ± 0.0056

📁 Archivos:
   data/05_modelado/results/results_otros.parquet
   data/05_modelado/results/results_otros.xlsx
   data/05_modelado/results/results_otros.json
   data/05_modelado/models/  (9 × .pkl)
   docs/html/fase5/m04_otros.html

➡️  Siguiente: f5_m05_mlp_ebm.ipynb
